This notebook computes the **Percent Trips Connected** metric following the
methodology of Mekuria, Furth & Nixon (2012), applied to the Belfast Local
Government District using the LTS-classified cycling network from the companion
notebook (`belfast_cycle_lts_assessment.ipynb`).

## Methodology Overview

Two points are said to be *connected* at a given LTS level if there exists a
path between them using only links that do not exceed that level of stress,
and the path does not involve undue detour. The detour criterion requires that
the low-stress path length must not exceed the shortest (all-network) path by
more than **25%**, or for short trips, **0.53 km (0.33 miles)**.

**Percent Trips Connected** is the share of total commuting demand (from Census
2021 OD data) that can be served by a connected, low-stress route at each LTS
threshold.

### Data Sources

| Dataset | Source | Spatial unit |
|---|---|---|
| LTS network | `belfast_cycle_lts_assessment.ipynb` output | Edge-level |
| OD commuting flows | PCTNI / Census 2021 ODWP01 | Super Data Zone (SDZ) |
| SDZ boundaries | PCTNI `zones_sdz.gpkg` | SDZ |

### Pipeline

1. Load LTS network, SDZ boundaries, and OD data
2. Filter OD pairs to Belfast LGD; exclude intra-zone trips
3. Construct routable graph from LTS network
4. Snap SDZ centroids to nearest network node
5. For each LTS threshold (1–4), compute connectivity for all OD pairs
6. Calculate Percent Trips Connected and Percent Nodes Connected
7. Visualise results

## Environment Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from shapely.geometry import Point
from shapely.ops import nearest_points

# Network analysis — igraph is much faster than networkx for large graphs
import igraph as ig
import networkx as nx

from collections import defaultdict
from itertools import combinations

# Project paths
DATA_DIR = Path('../Data')
OUTPUT_DIR = Path('../Figure')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Coordinate reference systems
CRS_WGS84 = 'EPSG:4326'
CRS_IG    = 'EPSG:29903'   # Irish Grid — all spatial ops use this

print('Environment ready.')


Environment ready.


## Load Data

### LTS Network

Load the LTS-classified cycling network produced by the companion notebook.
Each edge has an `lts` column (1–4) and geometry in Irish Grid.

In [2]:
# ── Load the LTS-classified network ──────────────────────────────────────────
# This parquet file is the output of belfast_cycle_lts_assessment.ipynb
edges = gpd.read_parquet(DATA_DIR / 'LTS_Belfast.parquet')
edges = edges.to_crs(CRS_IG)

print(f'Total edges: {len(edges):,}')
print(f'CRS: {edges.crs}')
print(f'\nLTS distribution:')
print(edges['lts'].value_counts().sort_index())
print(f'\nNaN LTS: {edges["lts"].isna().sum():,}')


Total edges: 23,464
CRS: EPSG:29903

LTS distribution:
lts
1.0    9752
2.0    9886
3.0    1528
4.0    2298
Name: count, dtype: int64

NaN LTS: 0


### Study Area — Belfast LGD Zones

Load SDZ boundaries and filter to Belfast LGD.

In [3]:
# ── SDZ zones ────────────────────────────────────────────────────────────────
sdz_all = gpd.read_file(DATA_DIR / 'zones_sdz.gpkg', engine='pyogrio')
sdz_all = sdz_all.to_crs(CRS_IG)

# Filter to Belfast LGD
belfast_sdz = sdz_all[sdz_all['lgd2014_nm'] == 'Belfast'].copy()
belfast_boundary = belfast_sdz.unary_union

print(f'Belfast SDZ count: {len(belfast_sdz)}')
print(f'Belfast area: {belfast_sdz.geometry.area.sum() / 1e6:.1f} km²')


Belfast SDZ count: 175
Belfast area: 137.7 km²


### OD Commuting Data

Load the Census 2021 OD flows. The PCTNI filtered version retains only
`place_of_work_ind_code == 4` (fixed workplace within UK), with both
origin and destination coded at SDZ level.

In [4]:
# ── OD data ──────────────────────────────────────────────────────────────────
od_raw = pd.read_csv(DATA_DIR / 'od_ni_open_filtered.csv')

print(f'Total OD records: {len(od_raw):,}')
print(f'Total commuters: {od_raw["count"].sum():,}')
print(f'Columns: {list(od_raw.columns)}')
od_raw.head()


Total OD records: 108,656
Total commuters: 545,210
Columns: ['area_of_residence_code', 'workplace_area_code', 'place_of_work_ind_code', 'count']


,area_of_residence_code,workplace_area_code,place_of_work_ind_code,count
0,N21000001,N21000001,4,51
1,N21000001,N21000002,4,3
2,N21000001,N21000003,4,4
3,N21000001,N21000004,4,34
4,N21000001,N21000005,4,28


## OD Data Preparation

The Census 2021 OD commuting flows (ODWP01) are recorded at Super Data Zone
(SDZ) level — the finest geography available in this dataset. Since the OD
codes match `zones_sdz.gpkg` directly, no spatial aggregation is required.
This section filters the OD pairs to those with both origin and destination
within Belfast LGD, and excludes intra-zone trips following Mekuria et al.
(2012) to limit the influence of very short trips for which walking is the
dominant mode.

In [5]:
# ── Identify SDZ code column ────────────────────────────────────────────────
sdz_code_col = [c for c in belfast_sdz.columns if 'sdz' in c.lower() and 'code' in c.lower()]
if not sdz_code_col:
    sdz_code_col = belfast_sdz.columns[0]  # fallback
else:
    sdz_code_col = sdz_code_col[0]
print(f'Using SDZ code column: "{sdz_code_col}"')

# ── Filter OD to Belfast: both origin AND destination within Belfast LGD ────
belfast_sdz_codes = set(belfast_sdz[sdz_code_col].values)

od = od_raw[
    od_raw['area_of_residence_code'].isin(belfast_sdz_codes) &
    od_raw['workplace_area_code'].isin(belfast_sdz_codes)
].copy()

od = od.rename(columns={
    'area_of_residence_code': 'origin_sdz',
    'workplace_area_code': 'dest_sdz',
})

# Exclude intra-zone trips (Mekuria et al. 2012)
od = od[od['origin_sdz'] != od['dest_sdz']].copy()

print(f'Belfast inter-SDZ OD pairs: {len(od):,}')
print(f'Total inter-SDZ commuters: {od["count"].sum():,}')
print(f'Unique origin SDZs: {od["origin_sdz"].nunique()}')
print(f'Unique destination SDZs: {od["dest_sdz"].nunique()}')


Using SDZ code column: "sdz2021_cd"
Belfast inter-SDZ OD pairs: 13,412
Total inter-SDZ commuters: 68,877
Unique origin SDZs: 175
Unique destination SDZs: 175


## Build Routable Graph

Construct an igraph graph from the LTS network edges. Each edge carries its
length (metres) and LTS value. Nodes are created from unique endpoint
coordinates rounded to 1 m precision.

In [22]:
def edges_to_igraph(edges_gdf: gpd.GeoDataFrame, snap_tolerance: float = 5.0) -> tuple[ig.Graph, dict]:
    """
    Convert edge GeoDataFrame into an igraph Graph.
    Uses KD-tree snapping to merge nearby endpoints into shared vertices.
    
    Parameters
    ----------
    snap_tolerance : float
        Endpoints within this distance (metres) are merged into one vertex.
    """
    from scipy.spatial import cKDTree
    
    # Step 1: Extract all endpoints
    all_points = []
    edge_data = []
    
    for _, row in edges_gdf.iterrows():
        geom = row.geometry
        lts = row.get('lts', np.nan)
        if pd.isna(lts):
            continue
        
        if geom.geom_type == 'MultiLineString':
            start = list(geom.geoms[0].coords)[0]
            end = list(geom.geoms[-1].coords)[-1]
        else:
            coords = list(geom.coords)
            start = coords[0]
            end = coords[-1]
        
        start = (start[0], start[1])
        end = (end[0], end[1])
        
        if abs(start[0] - end[0]) < 0.01 and abs(start[1] - end[1]) < 0.01:
            continue
        
        idx_s = len(all_points)
        all_points.append(start)
        idx_e = len(all_points)
        all_points.append(end)
        edge_data.append((idx_s, idx_e, geom.length, int(lts)))
    
    # Step 2: Cluster nearby points using KD-tree
    pts = np.array(all_points)
    tree = cKDTree(pts)
    
    # Assign each point to a cluster (vertex ID)
    visited = np.full(len(pts), -1, dtype=int)
    vid_counter = 0
    
    for i in range(len(pts)):
        if visited[i] >= 0:
            continue
        # Find all points within snap_tolerance
        neighbours = tree.query_ball_point(pts[i], snap_tolerance)
        for nb in neighbours:
            if visited[nb] < 0:
                visited[nb] = vid_counter
        vid_counter += 1
    
    # Step 3: Build graph
    edge_list = []
    lengths = []
    lts_vals = []
    
    for idx_s, idx_e, length, lts in edge_data:
        v_s = visited[idx_s]
        v_e = visited[idx_e]
        if v_s != v_e:
            edge_list.append((v_s, v_e))
            lengths.append(length)
            lts_vals.append(lts)
    
    G = ig.Graph(n=vid_counter, edges=edge_list, directed=False)
    G.es['length'] = lengths
    G.es['lts'] = lts_vals
    
    # Build coord_to_vid: use cluster centroid as representative coordinate
    cluster_coords = defaultdict(list)
    for i, vid in enumerate(visited):
        cluster_coords[vid].append(pts[i])
    
    coord_to_vid = {}
    vid_x = [0.0] * vid_counter
    vid_y = [0.0] * vid_counter
    for vid, coord_list in cluster_coords.items():
        cx = np.mean([c[0] for c in coord_list])
        cy = np.mean([c[1] for c in coord_list])
        vid_x[vid] = cx
        vid_y[vid] = cy
        coord_to_vid[(round(cx, 1), round(cy, 1))] = vid
    
    G.vs['x'] = vid_x
    G.vs['y'] = vid_y
    
    return G, coord_to_vid


In [23]:
G_full, coord_to_vid = edges_to_igraph(edges, snap_tolerance=5.0)
components = G_full.connected_components()
sizes = sorted(components.sizes(), reverse=True)
print(f'Components: {len(components)}')
print(f'Largest: {sizes[0]:,} / {G_full.vcount():,} ({sizes[0]/G_full.vcount()*100:.1f}%)')


Components: 12522
Largest: 4,454 / 34,667 (12.8%)


In [24]:
import osmnx as ox

ox.settings.use_cache = True
ox.settings.log_console = False

# ── Step 1: Extract Belfast road network via OSMnx ──────────────────────────
# Use the dissolved Belfast boundary polygon
belfast_poly_4326 = belfast_sdz.to_crs('EPSG:4326').unary_union

G_osm = ox.graph_from_polygon(
    belfast_poly_4326,
    network_type='all',
    simplify=True,
    retain_all=False
)

print(f'OSMnx graph — nodes: {G_osm.number_of_nodes():,}, edges: {G_osm.number_of_edges():,}')
print(f'Connected: {nx.is_connected(G_osm.to_undirected())}')

# ── Step 2: Convert to GeoDataFrames ────────────────────────────────────────
nodes_ox, edges_ox = ox.graph_to_gdfs(G_osm)
edges_ox = edges_ox.to_crs(CRS_IG)
edges_ox['length_m'] = edges_ox.geometry.length

print(f'Edges GeoDataFrame: {len(edges_ox):,}')
print(f'Edges have u/v index: {edges_ox.index.names}')

OSMnx graph — nodes: 42,542, edges: 108,869
Connected: True
Edges GeoDataFrame: 108,869
Edges have u/v index: ['u', 'v', 'key']


In [25]:
# ── Step 3: Match LTS values to OSMnx edges via spatial join ────────────────

# Prepare LTS edges — only keep edges with valid LTS
lts_edges = edges[edges['lts'].notna()][['geometry', 'lts', 'highway', 'cycle_infra']].copy()
lts_edges = lts_edges.to_crs(CRS_IG)

# Buffer OSMnx edges slightly for matching (5m buffer)
edges_ox_match = edges_ox[['geometry']].copy()
edges_ox_match['ox_idx'] = range(len(edges_ox_match))

# Use representative point of each OSMnx edge to find nearest LTS edge
edges_ox_match['midpoint'] = edges_ox_match.geometry.interpolate(0.5, normalized=True)
mid_gdf = gpd.GeoDataFrame(
    edges_ox_match[['ox_idx']], 
    geometry=edges_ox_match['midpoint'], 
    crs=CRS_IG
)

# Buffer LTS edges and spatial join
lts_buffered = lts_edges.copy()
lts_buffered['geometry'] = lts_buffered.geometry.buffer(10)  # 10m buffer

joined = gpd.sjoin(mid_gdf, lts_buffered, how='left', predicate='within')

# For edges that matched multiple LTS edges, take the one with highest LTS (conservative)
joined_dedup = joined.groupby('ox_idx').agg({
    'lts': 'first',  # take first match; could also use 'max' for conservative
    'highway': 'first',
    'cycle_infra': 'first',
}).reset_index()

# Assign LTS to OSMnx edges
edges_ox = edges_ox.reset_index()
edges_ox['lts'] = joined_dedup.set_index('ox_idx')['lts'].values

matched = edges_ox['lts'].notna().sum()
total = len(edges_ox)
print(f'LTS matched: {matched:,} / {total:,} ({matched/total*100:.1f}%)')
print(f'Unmatched: {total - matched:,}')
print(f'\nLTS distribution:')
print(edges_ox['lts'].value_counts().sort_index())

LTS matched: 102,107 / 108,869 (93.8%)
Unmatched: 6,762

LTS distribution:
lts
1.0    42901
2.0    33826
3.0    10759
4.0    14621
Name: count, dtype: int64


In [26]:
# ── Step 4: Fallback for unmatched edges ────────────────────────────────────
FALLBACK_LTS = {
    'cycleway': 1, 'path': 1, 'track': 1, 'living_street': 1,
    'footway': 1, 'pedestrian': 1, 'bridleway': 1,
    'residential': 1, 'service': 2,
    'unclassified': 2, 'tertiary': 2, 'tertiary_link': 2,
    'secondary': 3, 'secondary_link': 3, 'steps': 3,
    'primary': 4, 'primary_link': 4, 'trunk': 4, 'trunk_link': 4,
}

mask = edges_ox['lts'].isna()
# OSMnx highway column can be a list — take first element
edges_ox.loc[mask, 'lts'] = edges_ox.loc[mask, 'highway'].apply(
    lambda h: FALLBACK_LTS.get(h[0] if isinstance(h, list) else h, 3)
)

print(f'Still unmatched: {edges_ox["lts"].isna().sum()}')
print(f'\nFinal LTS distribution:')
print(edges_ox['lts'].value_counts().sort_index())

Still unmatched: 0

Final LTS distribution:
lts
1.0    49273
2.0    33980
3.0    10965
4.0    14651
Name: count, dtype: int64


In [27]:
# ── Step 5: Build igraph from OSMnx edges (using u/v node IDs) ──────────────
node_ids = list(set(edges_ox['u'].tolist() + edges_ox['v'].tolist()))
node_id_to_vid = {nid: i for i, nid in enumerate(node_ids)}

edge_list = [
    (node_id_to_vid[row.u], node_id_to_vid[row.v])
    for row in edges_ox.itertuples()
]

G_full = ig.Graph(n=len(node_ids), edges=edge_list, directed=False)
G_full.es['length'] = edges_ox['length_m'].tolist()
G_full.es['lts'] = edges_ox['lts'].astype(int).tolist()

# Store coordinates for snapping
G_full.vs['x'] = [nodes_ox.loc[nid, 'x'] for nid in node_ids]
G_full.vs['y'] = [nodes_ox.loc[nid, 'y'] for nid in node_ids]

# Update coord_to_vid for centroid snapping (use Irish Grid)
nodes_ig = nodes_ox.to_crs(CRS_IG)
coord_to_vid = {
    (nodes_ig.loc[nid, 'geometry'].x, nodes_ig.loc[nid, 'geometry'].y): node_id_to_vid[nid]
    for nid in node_ids
}

# Verify
comps = G_full.connected_components()
print(f'Components: {len(comps)}')
print(f'Vertices: {G_full.vcount():,}, Edges: {G_full.ecount():,}')

Components: 1
Vertices: 42,542, Edges: 108,869


In [28]:
sdz_to_vid = snap_points_to_graph(belfast_sdz, coord_to_vid, sdz_code_col)
results = compute_connectivity(G_full, od, sdz_to_vid)

Snapped 175 zones to network
Snap distance — median: 28 m, max: 591 m, mean: 41 m
Computing shortest paths on full network...
Valid OD pairs (0.5–6 miles): 12,132
Total demand in range: 63,913

--- LTS ≤ 1 ---
  Sub-graph edges: 49,273 / 108,869
  Pairs connected: 42 / 12,132 (0.3%)
  Trips connected: 249 / 63,913 (0.4%)

--- LTS ≤ 2 ---
  Sub-graph edges: 83,253 / 108,869
  Pairs connected: 232 / 12,132 (1.9%)
  Trips connected: 1,130 / 63,913 (1.8%)

--- LTS ≤ 3 ---
  Sub-graph edges: 94,218 / 108,869
  Pairs connected: 1,826 / 12,132 (15.1%)
  Trips connected: 8,519 / 63,913 (13.3%)

--- LTS ≤ 4 ---
  Sub-graph edges: 108,869 / 108,869
  Pairs connected: 12,132 / 12,132 (100.0%)
  Trips connected: 63,913 / 63,913 (100.0%)


## Snap SDZ Centroids to Network

Each SDZ centroid is snapped to the nearest graph vertex. This provides the
network entry/exit point for routing.

In [16]:
def snap_points_to_graph(points_gdf, coord_to_vid, code_col):
    """
    Snap zone centroids to nearest graph vertex.

    Returns dict: zone_code → vertex_id
    """
    from scipy.spatial import cKDTree

    coords = np.array(list(coord_to_vid.keys()))
    vids = np.array(list(coord_to_vid.values()))
    tree = cKDTree(coords)

    zone_to_vid = {}
    snap_distances = []

    for _, row in points_gdf.iterrows():
        centroid = row.geometry.centroid
        pt = np.array([centroid.x, centroid.y])
        dist, idx = tree.query(pt)
        zone_to_vid[row[code_col]] = int(vids[idx])
        snap_distances.append(dist)

    snap_distances = np.array(snap_distances)
    print(f'Snapped {len(zone_to_vid)} zones to network')
    print(f'Snap distance — median: {np.median(snap_distances):.0f} m, '
          f'max: {np.max(snap_distances):.0f} m, '
          f'mean: {np.mean(snap_distances):.0f} m')

    return zone_to_vid


In [17]:
sdz_to_vid = snap_points_to_graph(belfast_sdz, coord_to_vid, sdz_code_col)

# Check how many OD SDZs are successfully snapped
od_origins = set(od['origin_sdz'])
od_dests = set(od['dest_sdz'])
all_od_sdz = od_origins | od_dests
snapped = all_od_sdz & set(sdz_to_vid.keys())
print(f'\nOD SDZs in network: {len(snapped)} / {len(all_od_sdz)}')


Snapped 175 zones to network
Snap distance — median: 28 m, max: 591 m, mean: 43 m

OD SDZs in network: 175 / 175


## Connectivity Analysis

### Core Algorithm

For each LTS threshold (1, 2, 3, 4):

1. Extract a **sub-graph** containing only edges with LTS ≤ threshold.
2. For each OD pair, compute the shortest path on the **sub-graph**.
3. Compare against the shortest path on the **full graph** (all edges).
4. An OD pair is **connected** if:
   - A path exists on the sub-graph, AND
   - The detour ratio ≤ 1.25 (or absolute detour ≤ 530 m for short trips).

The detour thresholds follow Mekuria et al. (2012): the low-stress path
should not exceed the most direct route by more than 25%, or for short
trips, 0.33 miles (≈ 530 m).

In [18]:
# ── Distance thresholds (Mekuria et al. 2012) ─────────────────────────────
DETOUR_RATIO     = 1.25       # max 25% longer than shortest path
DETOUR_ABS_M     = 530.0      # 0.33 miles in metres — for short trips
MIN_TRIP_M       = 805.0      # 0.5 miles — exclude very short trips
MAX_TRIP_M       = 9_656.0    # 6 miles — exclude very long trips


def compute_connectivity(
    G_full: ig.Graph,
    od_pairs: pd.DataFrame,
    sdz_to_vid: dict,
    lts_thresholds: list[int] = [1, 2, 3, 4],
) -> pd.DataFrame:
    """
    Compute Percent Trips Connected for each LTS threshold.

    Parameters
    ----------
    G_full : ig.Graph
        Full network graph with 'length' and 'lts' edge attributes.
    od_pairs : pd.DataFrame
        Columns: origin_sdz, dest_sdz, count.
    sdz_to_vid : dict
        SDZ code → graph vertex ID.
    lts_thresholds : list
        LTS levels to evaluate.

    Returns
    -------
    results : pd.DataFrame
        Per-OD pair connectivity at each LTS threshold.
    """
    # ── Step 1: Precompute shortest paths on full network ─────────────────
    print('Computing shortest paths on full network...')

    valid_pairs = od_pairs[
        od_pairs['origin_sdz'].isin(sdz_to_vid) &
        od_pairs['dest_sdz'].isin(sdz_to_vid)
    ].copy()

    valid_pairs['origin_vid'] = valid_pairs['origin_sdz'].map(sdz_to_vid)
    valid_pairs['dest_vid'] = valid_pairs['dest_sdz'].map(sdz_to_vid)

    # Remove pairs where origin and destination snap to the same vertex
    valid_pairs = valid_pairs[valid_pairs['origin_vid'] != valid_pairs['dest_vid']].copy()

    unique_origins = valid_pairs['origin_vid'].unique().tolist()

    # Full-network shortest paths from each unique origin
    full_sp = {}
    for o_vid in unique_origins:
        dists = G_full.shortest_paths(source=o_vid, weights='length')[0]
        full_sp[o_vid] = dists

    # Attach full-network distance to each OD pair
    valid_pairs['dist_full'] = [
        full_sp[row.origin_vid][row.dest_vid]
        for row in valid_pairs.itertuples()
    ]

    # Filter by distance range (Mekuria: 0.5–6 miles)
    valid_pairs = valid_pairs[
        (valid_pairs['dist_full'] >= MIN_TRIP_M) &
        (valid_pairs['dist_full'] <= MAX_TRIP_M) &
        (valid_pairs['dist_full'] < float('inf'))
    ].copy()

    print(f'Valid OD pairs (0.5–6 miles): {len(valid_pairs):,}')
    print(f'Total demand in range: {valid_pairs["count"].sum():,}')

    # ── Step 2: For each LTS threshold, build sub-graph and check ────────
    for lts_max in lts_thresholds:
        print(f'\n--- LTS ≤ {lts_max} ---')

        # Extract sub-graph: keep only edges with LTS ≤ threshold
        sub_edge_ids = [e.index for e in G_full.es if e['lts'] <= lts_max]
        G_sub = G_full.subgraph_edges(sub_edge_ids, delete_vertices=False)

        print(f'  Sub-graph edges: {G_sub.ecount():,} / {G_full.ecount():,}')

        # Shortest paths on sub-graph from each unique origin
        sub_sp = {}
        for o_vid in unique_origins:
            dists = G_sub.shortest_paths(source=o_vid, weights='length')[0]
            sub_sp[o_vid] = dists

        # Check connectivity with detour criterion
        col = f'connected_lts{lts_max}'
        connected_flags = []
        for row in valid_pairs.itertuples():
            d_sub = sub_sp[row.origin_vid][row.dest_vid]
            d_full = row.dist_full

            if d_sub == float('inf'):
                connected_flags.append(False)
            else:
                # Detour criterion: max(25% longer, or 530m absolute)
                max_allowed = max(d_full * DETOUR_RATIO, d_full + DETOUR_ABS_M)
                connected_flags.append(d_sub <= max_allowed)

        valid_pairs[col] = connected_flags

        n_connected = sum(connected_flags)
        trips_connected = valid_pairs.loc[valid_pairs[col], 'count'].sum()
        pct_pairs = n_connected / len(valid_pairs) * 100
        pct_trips = trips_connected / valid_pairs['count'].sum() * 100

        print(f'  Pairs connected: {n_connected:,} / {len(valid_pairs):,} ({pct_pairs:.1f}%)')
        print(f'  Trips connected: {trips_connected:,} / {valid_pairs["count"].sum():,} ({pct_trips:.1f}%)')

    return valid_pairs


In [19]:
results = compute_connectivity(G_full, od, sdz_to_vid)


Computing shortest paths on full network...
Valid OD pairs (0.5–6 miles): 109
Total demand in range: 673

--- LTS ≤ 1 ---
  Sub-graph edges: 9,675 / 23,184
  Pairs connected: 0 / 109 (0.0%)
  Trips connected: 0 / 673 (0.0%)

--- LTS ≤ 2 ---
  Sub-graph edges: 19,376 / 23,184
  Pairs connected: 0 / 109 (0.0%)
  Trips connected: 0 / 673 (0.0%)

--- LTS ≤ 3 ---
  Sub-graph edges: 20,890 / 23,184
  Pairs connected: 3 / 109 (2.8%)
  Trips connected: 8 / 673 (1.2%)

--- LTS ≤ 4 ---
  Sub-graph edges: 23,184 / 23,184
  Pairs connected: 109 / 109 (100.0%)
  Trips connected: 673 / 673 (100.0%)


In [13]:
# 1. 检查有多少OD对在full graph上就不连通
valid = od[
    od['origin_sdz'].isin(sdz_to_vid) &
    od['dest_sdz'].isin(sdz_to_vid)
].copy()
valid['o_vid'] = valid['origin_sdz'].map(sdz_to_vid)
valid['d_vid'] = valid['dest_sdz'].map(sdz_to_vid)

inf_count = 0
short_count = 0
long_count = 0
good_count = 0

for row in valid.itertuples():
    d = G_full.shortest_paths(source=row.o_vid, target=row.d_vid, weights='length')[0][0]
    if d == float('inf'):
        inf_count += 1
    elif d < 805:
        short_count += 1
    elif d > 9656:
        long_count += 1
    else:
        good_count += 1

print(f'Not connected (inf): {inf_count}')
print(f'Too short (<0.5mi):  {short_count}')
print(f'Too long (>6mi):     {long_count}')
print(f'In range:            {good_count}')

Not connected (inf): 13289
Too short (<0.5mi):  2
Too long (>6mi):     12
In range:            109


In [20]:
components = G_full.connected_components()
print(f'Connected components: {len(components)}')
sizes = sorted(components.sizes(), reverse=True)
print(f'Top 10 component sizes: {sizes[:10]}')
print(f'Largest component: {sizes[0]:,} / {G_full.vcount():,} ({sizes[0]/G_full.vcount()*100:.1f}%)')


Connected components: 13532
Top 10 component sizes: [4125, 55, 37, 29, 26, 24, 24, 21, 21, 20]
Largest component: 4,125 / 36,150 (11.4%)


In [21]:
print(edges.columns.tolist())
# 看有没有 u, v, osmid, from, to 这类列

['access', 'bicycle', 'bridge', 'busway', 'cycleway', 'foot', 'footway', 'highway', 'int_ref', 'junction', 'lanes', 'lit', 'maxspeed', 'motorcar', 'motorroad', 'motor_vehicle', 'name', 'oneway', 'psv', 'ref', 'service', 'segregated', 'sidewalk', 'smoothness', 'surface', 'tracktype', 'tunnel', 'turn', 'width', 'cycleway:left', 'cycleway:right', 'cycleway:both', 'id', 'timestamp', 'version', 'tags', 'osm_type', 'geometry', 'length', 'cycle_infra', 'lts', 'lts_status']


## Summary Metrics

Compute and tabulate the two Mekuria indicators:

- **Percent Trips Connected**: weighted by commuting flow (demand)
- **Percent Nodes Connected**: unweighted (share of OD pairs)

In [ ]:
# ── Build summary table ──────────────────────────────────────────────────────
total_trips = results['count'].sum()
total_pairs = len(results)

summary_rows = []
for lts_max in [1, 2, 3, 4]:
    col = f'connected_lts{lts_max}'
    trips_conn = results.loc[results[col], 'count'].sum()
    pairs_conn = results[col].sum()

    summary_rows.append({
        'LTS Threshold': f'LTS ≤ {lts_max}',
        'Pairs Connected': int(pairs_conn),
        'Total Pairs': total_pairs,
        '% Nodes Connected': round(pairs_conn / total_pairs * 100, 1),
        'Trips Connected': int(trips_conn),
        'Total Trips': int(total_trips),
        '% Trips Connected': round(trips_conn / total_trips * 100, 1),
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))


## Visualisation

### Percent Trips Connected & Percent Nodes Connected

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

lts_labels = ['LTS ≤ 1', 'LTS ≤ 2', 'LTS ≤ 3', 'LTS ≤ 4']
lts_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#c0392b']

# ── Percent Trips Connected ──────────────────────────────────────────────────
ax = axes[0]
pct_trips = summary['% Trips Connected'].values
bars = ax.bar(lts_labels, pct_trips, color=lts_colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('% Trips Connected', fontsize=13)
ax.set_title('Percent Trips Connected', fontsize=15, fontweight='bold')
ax.set_ylim(0, 105)
for bar, val in zip(bars, pct_trips):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

# ── Percent Nodes Connected ──────────────────────────────────────────────────
ax = axes[1]
pct_nodes = summary['% Nodes Connected'].values
bars = ax.bar(lts_labels, pct_nodes, color=lts_colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('% Nodes Connected', fontsize=13)
ax.set_title('Percent Nodes Connected', fontsize=15, fontweight='bold')
ax.set_ylim(0, 105)
for bar, val in zip(bars, pct_nodes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Belfast Cycling Network Connectivity (Mekuria et al. 2012)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'belfast_connectivity_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR / "belfast_connectivity_metrics.png"}')


### Per-SDZ Connectivity Score

For the dashboard, compute a per-SDZ metric: what share of outbound commuting
demand from each SDZ is connected at each LTS threshold. This allows
choropleth mapping at SDZ level.

In [ ]:
# ── Per-SDZ outbound connectivity score ──────────────────────────────────────
sdz_scores = []

for sdz_code in belfast_sdz[sdz_code_col].values:
    mask = results['origin_sdz'] == sdz_code
    subset = results[mask]
    if len(subset) == 0:
        row = {'sdz_code': sdz_code, 'total_trips': 0}
        for lts_max in [1, 2, 3, 4]:
            row[f'pct_trips_lts{lts_max}'] = np.nan
        sdz_scores.append(row)
        continue

    total = subset['count'].sum()
    row = {'sdz_code': sdz_code, 'total_trips': int(total)}
    for lts_max in [1, 2, 3, 4]:
        col = f'connected_lts{lts_max}'
        conn = subset.loc[subset[col], 'count'].sum()
        row[f'pct_trips_lts{lts_max}'] = round(conn / total * 100, 1) if total > 0 else 0.0
    sdz_scores.append(row)

sdz_scores_df = pd.DataFrame(sdz_scores)
print(sdz_scores_df.describe().round(1))


In [ ]:
# ── Choropleth: Per-SDZ % Trips Connected ────────────────────────────────────
plot_sdz = belfast_sdz.merge(
    sdz_scores_df, left_on=sdz_code_col, right_on='sdz_code', how='left'
)

fig, axes = plt.subplots(1, 4, figsize=(24, 7))

for i, lts_max in enumerate([1, 2, 3, 4]):
    ax = axes[i]
    col = f'pct_trips_lts{lts_max}'
    plot_sdz.plot(
        column=col, ax=ax, legend=True,
        cmap='RdYlGn', vmin=0, vmax=100,
        edgecolor='white', linewidth=0.3,
        missing_kwds={'color': '#cccccc', 'label': 'No data'},
        legend_kwds={'shrink': 0.6, 'label': '% Trips Connected'}
    )
    ax.set_title(f'LTS ≤ {lts_max}', fontsize=14, fontweight='bold')
    ax.set_axis_off()

plt.suptitle('Per-SDZ Outbound % Trips Connected — Belfast',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'belfast_sdz_connectivity_choropleth.png',
            dpi=200, bbox_inches='tight')
plt.show()


### Export Results for Dashboard

Save the per-SDZ scores and OD-level results for the interactive dashboard.

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
# Per-SDZ connectivity scores (for choropleth)
sdz_scores_df.to_csv(OUTPUT_DIR / 'belfast_sdz_connectivity_scores.csv', index=False)

# Summary table
summary.to_csv(OUTPUT_DIR / 'belfast_connectivity_summary.csv', index=False)

# OD-level results (for detailed analysis)
results.to_csv(OUTPUT_DIR / 'belfast_od_connectivity_results.csv', index=False)

print('Exported:')
print(f'  {OUTPUT_DIR / "belfast_sdz_connectivity_scores.csv"}')
print(f'  {OUTPUT_DIR / "belfast_connectivity_summary.csv"}')
print(f'  {OUTPUT_DIR / "belfast_od_connectivity_results.csv"}')


## References

- Mekuria, M. C., Furth, P. G. & Nixon, H. (2012). *Low-Stress Bicycling and
  Network Connectivity*. Mineta Transportation Institute Report 11-19.